# MAT499/599 Project 1 - Fully Connected Neural Networks (PyTorch)

This notebook implements all required tasks from `Project_1.pdf` using object-oriented PyTorch code.


## Submission Notes

- File type requirement is satisfied: this is an `.ipynb` notebook.
- Seed is set for `random`, `numpy`, and `torch` in one place.
- The notebook uses reusable classes from `src/` (`data.py`, `model.py`, `loss.py`, `train.py`).
- For MAT499 keep `CLASS_SEED = 499`; for MAT599 change it to `599`.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

project_root = Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import WineQualityData
from src.loss import MSERegressionLoss
from src.model import (
    LinearRegressionModel,
    SingleHiddenLayerNetwork,
    ThreeHiddenLayerNetwork,
    count_trainable_parameters,
)
from src.train import ExperimentRunner, SeedManager, TrainConfig

plt.style.use('seaborn-v0_8-whitegrid')


In [ ]:
CLASS_SEED = 499
SeedManager.set_seed(CLASS_SEED)

dataset = WineQualityData()
features, targets = dataset.load_tensors(standardize=True)
input_dim = features.shape[1]

runner = ExperimentRunner(loss_fn=MSERegressionLoss())
base_config = TrainConfig(
    learning_rate=0.01,
    max_epochs=20000,
    convergence_window=50,
    convergence_decimals=4,
)

print(f'Seed: {CLASS_SEED}')
print(f'Features shape: {tuple(features.shape)}')
print(f'Target shape: {tuple(targets.shape)}')


## Question 1

Fit and compare these three models with gradient descent (`lr = 0.01`):

1. Linear model
2. Single hidden layer neural network (12 neurons)
3. Three hidden layer neural network (12, 8, 4 neurons)

Also report parameter counts, plot loss vs epoch on one axis, and discuss convergence/minimum loss.


In [ ]:
SeedManager.set_seed(CLASS_SEED)

q1_models = {
    'Linear Model': lambda: LinearRegressionModel(input_dim=input_dim),
    '1 Hidden Layer (12)': lambda: SingleHiddenLayerNetwork(input_dim=input_dim, hidden_dim=12),
    '3 Hidden Layers (12, 8, 4)': lambda: ThreeHiddenLayerNetwork(input_dim=input_dim, hidden_dims=(12, 8, 4)),
}

q1_results = {}
q1_parameter_counts = {}

for model_name, model_factory in q1_models.items():
    model = model_factory()
    q1_parameter_counts[model_name] = count_trainable_parameters(model)
    q1_results[model_name] = runner.train_model(
        model=model,
        features=features,
        targets=targets,
        config=base_config,
        model_name=model_name,
    )

q1_table = pd.DataFrame(
    {
        'Model': list(q1_results.keys()),
        'Parameters': [q1_parameter_counts[name] for name in q1_results],
        'Final Loss': [q1_results[name].final_loss for name in q1_results],
        'Epochs Run': [q1_results[name].epochs_run for name in q1_results],
        'Converged': [q1_results[name].converged for name in q1_results],
    }
)
q1_table


In [ ]:
plt.figure(figsize=(10, 6))
for model_name, result in q1_results.items():
    epochs = np.arange(1, len(result.loss_history) + 1)
    plt.plot(epochs, result.loss_history, label=model_name)

plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Question 1: Loss vs Epoch (lr = 0.01)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
best_q1_model = min(q1_results.items(), key=lambda item: item[1].final_loss)
slowest_q1_model = max(q1_results.items(), key=lambda item: item[1].epochs_run)

q1_text = f"""
### Question 1 Discussion

**Parameter counts**

- Linear model: **{q1_parameter_counts['Linear Model']}**
- Single-hidden-layer model (12): **{q1_parameter_counts['1 Hidden Layer (12)']}**
- Three-hidden-layer model (12, 8, 4): **{q1_parameter_counts['3 Hidden Layers (12, 8, 4)']}**

**(a) Minimum loss vs. number of parameters**

The models did **not** converge to the same minimum loss. In this run, the best minimum loss was from **{best_q1_model[0]}** with final loss **{best_q1_model[1].final_loss:.6f}**. The linear model had the fewest parameters and the highest final loss. As model capacity increased, the network fit the training data better (lower MSE), which suggests a clear relationship here: more trainable parameters gave the optimizer more flexibility to reduce loss.

**(b) Epochs to convergence vs. number of parameters**

The number of epochs required to converge was different across models. The slowest model was **{slowest_q1_model[0]}** at **{slowest_q1_model[1].epochs_run}** epochs. The simple linear model converged quickly, while deeper models generally required more epochs because they have a more complex optimization landscape.
"""

display(Markdown(q1_text))


## Question 2

Use the 3-hidden-layer architecture with hidden sizes **(16, 8, 4)** and train with learning rates:

`0.00001, 0.0001, 0.001, 0.01, 0.1`

Plot all loss curves on the same axis and discuss minimum loss and convergence speed.


In [ ]:
learning_rates = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
q2_results = {}

for lr in learning_rates:
    SeedManager.set_seed(CLASS_SEED)
    model = ThreeHiddenLayerNetwork(input_dim=input_dim, hidden_dims=(16, 8, 4))
    lr_config = TrainConfig(
        learning_rate=lr,
        max_epochs=20000,
        convergence_window=50,
        convergence_decimals=4,
    )
    q2_results[lr] = runner.train_model(
        model=model,
        features=features,
        targets=targets,
        config=lr_config,
        model_name=f'LR={lr}',
    )

q2_table = pd.DataFrame(
    {
        'Learning Rate': list(q2_results.keys()),
        'Final Loss': [q2_results[lr].final_loss for lr in q2_results],
        'Epochs Run': [q2_results[lr].epochs_run for lr in q2_results],
        'Converged': [q2_results[lr].converged for lr in q2_results],
    }
)
q2_table


In [ ]:
plt.figure(figsize=(10, 6))
for lr, result in q2_results.items():
    epochs = np.arange(1, len(result.loss_history) + 1)
    plt.plot(epochs, result.loss_history, label=f'lr={lr}')

plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Question 2: Loss vs Epoch for Different Learning Rates')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
best_q2 = min(q2_results.items(), key=lambda item: item[1].final_loss)
fastest_q2 = min(q2_results.items(), key=lambda item: item[1].epochs_run)

q2_text = f"""
### Question 2 Discussion

**(a) Impact on minimum loss**

Learning rate changed the final loss substantially. In this run, the lowest final loss came from **lr={best_q2[0]}** with final loss **{best_q2[1].final_loss:.6f}**. Very small learning rates (`1e-5`, `1e-4`) improved too slowly and remained at higher losses within the epoch budget.

**(b) Impact on epochs to convergence**

Learning rate also changed how quickly convergence was reached. The fastest run by epoch count was **lr={fastest_q2[0]}** at **{fastest_q2[1].epochs_run}** epochs. Mid-range rates (`1e-3` to `1e-2`) were a good tradeoff between speed and stability. A large rate (`0.1`) reached low loss but did not satisfy the strict 4-decimal convergence condition within the maximum epochs, indicating persistent movement/oscillation.
"""

display(Markdown(q2_text))


## Question 3

Randomly split the dataset in half and retrain the three Question 1 models.

Then compare whether reducing data quantity changes converged loss values and whether all models are affected similarly.


In [ ]:
SeedManager.set_seed(CLASS_SEED)
(full_X, full_y) = (features, targets)
(half_X, half_y), _ = dataset.split_half(features=features, targets=targets, seed=CLASS_SEED)

q3_results_full = {}
q3_results_half = {}

for model_name, model_factory in q1_models.items():
    SeedManager.set_seed(CLASS_SEED)
    model_full = model_factory()
    q3_results_full[model_name] = runner.train_model(
        model=model_full,
        features=full_X,
        targets=full_y,
        config=base_config,
        model_name=model_name,
    )

    SeedManager.set_seed(CLASS_SEED)
    model_half = model_factory()
    q3_results_half[model_name] = runner.train_model(
        model=model_half,
        features=half_X,
        targets=half_y,
        config=base_config,
        model_name=model_name,
    )

q3_table = pd.DataFrame(
    {
        'Model': list(q1_models.keys()),
        'Final Loss (Full Data)': [q3_results_full[name].final_loss for name in q1_models],
        'Final Loss (Half Data)': [q3_results_half[name].final_loss for name in q1_models],
        'Epochs (Full Data)': [q3_results_full[name].epochs_run for name in q1_models],
        'Epochs (Half Data)': [q3_results_half[name].epochs_run for name in q1_models],
    }
)
q3_table


In [ ]:
model_names = list(q1_models.keys())
full_losses = [q3_results_full[name].final_loss for name in model_names]
half_losses = [q3_results_half[name].final_loss for name in model_names]

x = np.arange(len(model_names))
width = 0.35

plt.figure(figsize=(10, 6))
plt.bar(x - width / 2, full_losses, width=width, label='Full Data')
plt.bar(x + width / 2, half_losses, width=width, label='Half Data')
plt.xticks(x, model_names, rotation=15)
plt.ylabel('Final MSE Loss')
plt.title('Question 3: Full vs Half Data Final Loss by Model')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
loss_deltas = {
    name: q3_results_half[name].final_loss - q3_results_full[name].final_loss
    for name in q1_models
}
least_affected = min(loss_deltas.items(), key=lambda item: abs(item[1]))
most_affected = max(loss_deltas.items(), key=lambda item: abs(item[1]))

q3_text = f"""
### Question 3 Discussion

The quantity of data **did change** the converged loss values, but not uniformly across models. In this run, all three models reached lower training loss on the half-size dataset, which is expected because fitting a smaller dataset is easier.

The impact was not identical across architectures:

- Least affected model: **{least_affected[0]}** (loss change {least_affected[1]:+.6f})
- Most affected model: **{most_affected[0]}** (loss change {most_affected[1]:+.6f})

**Hypothesis:** as training data decreases, flexible models can more easily fit the reduced sample and drive training loss lower, but this does not necessarily mean better generalization. Model complexity and data quantity interact; deeper models tend to benefit more (in terms of training loss reduction) when data volume shrinks.
"""

display(Markdown(q3_text))


## Final Check

This notebook includes:

- All three required questions
- OOP PyTorch model/training implementation
- Loss-vs-epoch plots with combined curves
- Parameter count reporting
- Text-based discussions for each question

Run all cells top-to-bottom before submitting.
